## KYHPS 추적 상태 레이블링 (Tracking Status Labeling)
- 방법 A: 규칙 기반 분류 (Rule-Based)       ← Task2 기존 방법
- 방법 B: 점수 기반 탈락 위험도 분류         ← 신규 대안
- 방법 C: 응답 완성도 기반 품질 등급         ← 신규 대안

: 각 방법의 결과를 단일 DataFrame에 병합하여 비교 가능하게 출력.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ─────────────────────────────────────────────
# 0. 데이터 로딩 & 응답 지표 생성
# ─────────────────────────────────────────────
DATA_PATH = r".\2019-23_student_merged_wide_format.csv"
df = pd.read_csv(DATA_PATH, low_memory=False, encoding='cp949')

WAVES = ['w1', 'w2', 'w3', 'w4', 'w5']
for w in WAVES:
    df[f'responded_{w}'] = df[f'종단면 가중치_{w}'].notna().astype(int)

resp_cols = [f'responded_{w}' for w in WAVES]
df['waves_completed'] = df[resp_cols].sum(axis=1)
df['response_pattern'] = df[resp_cols].apply(
    lambda r: ''.join(r.astype(str)), axis=1
)

print(f"로딩 완료: {df.shape[0]:,}명 × {df.shape[1]:,}변수\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 방법 A: 규칙 기반 레이블링 (Rule-Based Classification)
# 기준: 종단 가중치 NULL 여부 → 응답 패턴 문자열로 분류
# ═══════════════════════════════════════════════════════════════
def assign_rule_based_label(row):
    """
    응답 패턴 문자열을 기반으로 추적 상태를 분류.

    Returns
    -------
    label      : str  — 추적 상태 레이블
    dropout_wave: int or None — 최초 탈락 파 번호
    attrition_type: str — 단조/불규칙/완전
    """
    pat = row['response_pattern']

    # [규칙 0] W1 미응답 — 이론상 불가, 오류로 처리
    if pat[0] == '0':
        return 'ERROR_W1_MISSING', None, 'error'

    # [규칙 1] 완전 5개년 응답
    if pat == '11111':
        return 'COMPLETE_5Y', None, 'complete'

    # [규칙 2] 단조 탈락 — '0' 첫 등장 이후 전부 0
    first_zero = pat.find('0')
    if first_zero != -1:
        tail = pat[first_zero:]
        if set(tail) == {'0'}:
            dropout_wave = first_zero + 1
            label_map = {2: 'DROPOUT_EARLY_W2',
                         3: 'DROPOUT_MID_W3',
                         4: 'DROPOUT_LATE_W4',
                         5: 'DROPOUT_LAST_W5'}
            return label_map.get(dropout_wave, f'DROPOUT_W{dropout_wave}'), dropout_wave, 'monotone'

    # [규칙 3] 불규칙 패턴 (재진입 또는 중간 결측)
    return 'IRREGULAR', None, 'irregular'


result_A = df[['설문번호', 'response_pattern', 'waves_completed']].copy()
labels = df.apply(assign_rule_based_label, axis=1, result_type='expand')
labels.columns = ['label_A', 'dropout_wave_A', 'attrition_type_A']
result_A = pd.concat([result_A, labels], axis=1)

print("=" * 65)
print("[ 방법 A: 규칙 기반 레이블링 결과 ]")
print("=" * 65)
dist_A = result_A['label_A'].value_counts()
for label, cnt in dist_A.items():
    print(f"  {label:<25}: {cnt:>5}명 ({cnt/len(df)*100:.1f}%)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 방법 B: 탈락 위험도 점수 기반 분류 (Propensity Score Tiering)
# 기준: 기저치(W1) 변수로 탈락 확률 추정 → 위험도 4분위 레이블
#
# 방법 A와의 차이점:
#   - 패턴의 '모양'이 아닌 '기저 특성'으로 분류
#   - 탈락 고위험 완전응답자, 저위험 탈락자를 식별 가능
#   - IPW 계산의 전 단계로 활용 가능
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("[ 방법 B: 탈락 위험도 점수 기반 분류 ]")
print("=" * 65)

# 기저치(W1) 예측 변수 선정
predictor_vars = [
    '학생 성별_w1',
    '평생담배제품사용경험률_w1',
    '우울감 경험_w1',
    '주관적 신체 이미지_w1',
    '하루 60분 이상 신체 활동 빈도_w1',
    '스트레스 경험(학업)_w1',
    '스트레스 경험(친구관계)_w1',
]
predictor_vars = [v for v in predictor_vars if v in df.columns]

target = (df['waves_completed'] < 5).astype(int)   # 1=탈락, 0=완전응답

sub_B = df[predictor_vars].copy()
sub_B = sub_B.fillna(sub_B.median(numeric_only=True))   # 단순 대입 (모델용)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(sub_B)

lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_scaled, target)
dropout_prob = lr.predict_proba(X_scaled)[:, 1]   # P(탈락)

df['dropout_prob_B'] = dropout_prob

# 4분위 기반 위험 등급
quartiles = pd.qcut(dropout_prob, q=4, labels=['LOW', 'MED_LOW', 'MED_HIGH', 'HIGH'])
df['label_B'] = quartiles.astype(str)

# 완전 응답 여부와 교차
df['is_completer'] = (df['waves_completed'] == 5).astype(int)
cross_B = pd.crosstab(df['label_B'], df['is_completer'],
                       rownames=['위험등급'], colnames=['완전응답(1)vs탈락(0)'])
cross_B.columns = ['탈락', '완전응답']
cross_B['탈락률(%)'] = (cross_B['탈락'] / (cross_B['탈락'] + cross_B['완전응답']) * 100).round(1)
print(cross_B)
print(f"\n  AUC 근사: {lr.score(X_scaled, target):.3f}  (정확도 기준)")
print("  ※ 방법 B는 CCA 분석 대상자 선별 또는 IPW 가중치 산출 기반으로 활용")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 방법 C: 응답 완성도 기반 품질 등급 (Data Quality Grading)
# 기준: ① 응답 파 수 ② 핵심 변수 결측 비율 ③ 역전 응답 여부
#       → 종합 품질 점수 → A/B/C/D 등급
#
# 방법 A·B와의 차이점:
#   - 탈락 여부만이 아닌 '응답의 질'을 반영
#   - 완전응답자 중에도 D등급 존재 가능 (핵심 변수 대량 결측/역전)
#   - 분석 포함 기준 결정 시 활용 (≥B등급만 포함 등)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("[ 방법 C: 응답 완성도 기반 품질 등급 ]")
print("=" * 65)

# 핵심 변수 정의 (분석 주제에 따라 조정 필요)
KEY_VARS_TEMPLATE = [
    '평생담배제품사용경험률_{w}',
    '평생 음주 경험(잔 기준)_{w}',
    '우울감 경험_{w}',
    '스트레스 경험(학업)_{w}',
]

all_key_vars = []
for w in WAVES:
    for tmpl in KEY_VARS_TEMPLATE:
        v = tmpl.format(w=w)
        if v in df.columns:
            all_key_vars.append(v)

# 점수 구성요소 (0~100점 만점)
# ① 응답 파 수 점수: 5파 완전=50점, 파당 10점
score_wave = df['waves_completed'] * 10   # max 50

# ② 핵심 변수 결측 비율 점수: 완전 응답=30점
key_miss_rate = df[all_key_vars].isnull().mean(axis=1)
score_key = (1 - key_miss_rate) * 30      # max 30

# ③ 음주 역전 페널티: 역전 1건당 -5점 (최대 -20점)
drink_vars = [f'평생 음주 경험(잔 기준)_{w}' for w in WAVES if f'평생 음주 경험(잔 기준)_{w}' in df.columns]
sub_d = df[drink_vars].copy()
for v in drink_vars:
    sub_d[v] = sub_d[v].map({1: 1, 2: 0})

reversal_count = pd.Series(0, index=df.index)
for i in range(1, len(drink_vars)):
    prev = sub_d[drink_vars[i-1]]
    curr = sub_d[drink_vars[i]]
    valid = prev.notna() & curr.notna()
    rev = ((prev == 1) & (curr == 0) & valid).astype(int)
    reversal_count += rev

score_consistency = (20 - (reversal_count * 5).clip(upper=20))   # max 20

# 종합 점수
df['quality_score_C'] = (score_wave + score_key + score_consistency).clip(upper=100)

# 등급 부여
def score_to_grade(s):
    if s >= 90: return 'A (우수)'
    elif s >= 70: return 'B (양호)'
    elif s >= 50: return 'C (보통)'
    else: return 'D (불량)'

df['label_C'] = df['quality_score_C'].apply(score_to_grade)

grade_dist = df['label_C'].value_counts().sort_index()
print(f"\n  {'등급':<12} {'N':>6} {'비율':>7}  {'평균점수':>10}")
print(f"  {'-'*40}")
for grade in ['A (우수)', 'B (양호)', 'C (보통)', 'D (불량)']:
    sub = df[df['label_C'] == grade]
    print(f"  {grade:<12} {len(sub):>6} {len(sub)/len(df)*100:>6.1f}%  "
          f"{sub['quality_score_C'].mean():>10.1f}점")

# 방법 C의 추가 가치: 완전응답자 중 D등급 존재 여부 확인
complete_low = df[(df['waves_completed'] == 5) & (df['label_C'] == 'D (불량)')]
print(f"\n  ※ 완전응답자({(df['waves_completed']==5).sum():,}명) 중 D등급: {len(complete_low)}명")
print("    → 방법 A로는 COMPLETE_5Y이지만 실제 분석 품질이 낮은 케이스 식별 가능")

In [ ]:
# ─────────────────────────────────────────────
# 최종 레이블 통합 DataFrame
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("[ 세 방법 통합 결과 샘플 (상위 10행) ]")
print("=" * 65)

final_df = df[['설문번호', 'response_pattern', 'waves_completed',
               'dropout_prob_B', 'quality_score_C']].copy()
final_df = final_df.merge(
    result_A[['설문번호', 'label_A', 'dropout_wave_A', 'attrition_type_A']],
    on='설문번호'
)
final_df['label_B'] = df['label_B'].values
final_df['label_C'] = df['label_C'].values

display_cols = ['설문번호', 'response_pattern', 'waves_completed',
                'label_A', 'label_B', 'quality_score_C', 'label_C']
print(final_df[display_cols].head(10).to_string(index=False))

# 출력 저장
OUT_PATH = r".\output_cleaning\tracking_labels_ABC.csv"
final_df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
print(f"\n  저장 완료: {OUT_PATH}")

print("\n" + "=" * 65)
print("[ 세 방법 비교 요약 ]")
print("=" * 65)
print("""
  방법 A: 규칙 기반
    장점 — 결정론적·재현 가능, 추가 모델 불필요
    단점 — '왜 탈락했는가'를 반영하지 못함
    용도 — 표본 기술통계, 완전 사례 집합 정의

  방법 B: 탈락 위험도 점수
    장점 — MAR 가정 하 IPW 전 단계, 고위험 완전응답자 식별
    단점 — 로지스틱 모델 가정에 의존, 기저변수 선정 민감
    용도 — IPW 가중치 산출, 민감도 분석 보조

  방법 C: 응답 품질 등급
    장점 — 탈락+결측+역전을 종합한 '데이터 품질' 지표
    단점 — 점수 가중 기준이 연구자 주관 개입
    용도 — 분석 포함/제외 기준 결정, 제한점 기술 근거
""")